# SimCLR Batch-Size Ablation (Nidhi)

This notebook runs the batch-size ablation for our SimCLR project.

**What it does:**
- Clones the project repo from GitHub
- Trains SimCLR with batch sizes 64, 128, 256, 512 (short schedule: 20 epochs, 20k images)
- Evaluates each with a linear probe
- Saves all results to `results/results.csv`
- Downloads the results CSV at the end

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → Run all

In [ ]:
# Step 1: Check GPU
!nvidia-smi

In [ ]:
# Step 2: Clone the repo
import os

REPO_URL = "https://github.com/ShubhamKang/SimCLR-Deep-Learning.git"
REPO_DIR = "/content/SimCLR-Deep-Learning"

if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}

!git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Step 3: Install dependencies
!pip install -q torch torchvision tqdm pandas matplotlib

In [ ]:
# Step 4: Check everything is in place
!ls

In [ ]:
# Step 5: Run batch-size ablation
# 20 epochs, 20k image subset — same as augmentation ablation for fair comparison
# Runs 4 pretrainings + 4 linear probe evaluations back to back
# Total time: roughly 25-45 minutes on a T4 GPU

import time

EPOCHS = 20
SUBSET = 20000
DATA_DIR = "/content/data"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("results", exist_ok=True)

timing = {}  # record time per batch size

for bs in [64, 128, 256, 512]:
    print(f"\n{'='*60}")
    print(f"BATCH SIZE: {bs}")
    print(f"{'='*60}")

    t_start = time.time()

    # Pretrain
    ret = os.system(
        f"python train_simclr.py "
        f"--name bs_{bs} "
        f"--epochs {EPOCHS} "
        f"--subset {SUBSET} "
        f"--batch_size {bs} "
        f"--data {DATA_DIR} "
        f"--workers 2"
    )
    if ret != 0:
        print(f"ERROR: pretraining failed for bs={bs}")
        break

    t_pretrain = time.time() - t_start

    # Linear probe evaluation
    ret = os.system(
        f"python evaluate.py "
        f"--mode linear "
        f"--ckpt checkpoints/bs_{bs}.pt "
        f"--frac 1.0 "
        f"--tag bs_{bs} "
        f"--data {DATA_DIR}"
    )
    if ret != 0:
        print(f"ERROR: evaluation failed for bs={bs}")
        break

    t_total = time.time() - t_start
    timing[bs] = {"pretrain_sec": int(t_pretrain), "total_sec": int(t_total)}
    print(f"bs={bs} done | pretrain: {t_pretrain/60:.1f} min | total: {t_total/60:.1f} min")

print("\n=== TIMING SUMMARY ===")
for bs, t in timing.items():
    print(f"bs={bs}: pretrain={t['pretrain_sec']//60}m{t['pretrain_sec']%60}s, total={t['total_sec']//60}m{t['total_sec']%60}s")

In [ ]:
# Step 6: Check GPU memory info (note this down for the report)
!nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free --format=csv

In [ ]:
# Step 7: Show results
import pandas as pd

df = pd.read_csv("results/results.csv")
bs_results = df[df["tag"].str.startswith("bs_")].copy()
bs_results["batch_size"] = bs_results["tag"].str[3:].astype(int)
bs_results = bs_results.sort_values("batch_size")
bs_results["test_acc_%"] = (bs_results["test_acc"].astype(float) * 100).round(2)
print(bs_results[["batch_size", "test_acc_%"]].to_string(index=False))

In [ ]:
# Step 8: Download results.csv to your computer
# This is the file you will commit back to the repo
from google.colab import files
files.download("results/results.csv")

## After this notebook finishes

1. The `results.csv` download will start automatically in Step 8.
2. Copy the downloaded file into your local repo at `results/results.csv` (it already has Sandhya's supervised baseline rows — this adds your bs_* rows).
3. Note down the timing numbers from Step 5 and the GPU name from Step 6 — you'll need them for the batch-size analysis section of the report.
4. Come back here and tell me the results — I'll write the analysis section.